# 13b · Cross-resolution test: the two repeat runs

The main run (notebook 13) finished, but the files of its last two parts were lost when the Kaggle session closed.
This notebook makes **only those two files** again, with the same model, prompt and settings:

| Run | Locations | What it shows | Time |
|---|---|---|---|
| `missouri_native_test150` | 150 | original Missouri images again: do today's answers match the 6 Nov run? | ~13 min |
| `hisar_2.5m_rerun` | 300 | original (super-resolved) Hisar images again: same check for Hisar | ~25 min |

About **45 minutes** in total, including installing Ollama and downloading the model.

**How to run so the files are KEPT (important):**
1. Right panel → **Accelerator: GPU T4 x2**, **Internet: On**, **Add Input → `gully-collages`**.
2. Top right → **Save Version** → choose **Save & Run All (Commit)** → **Save**.
   Do **not** press the ▶ buttons in the editor. You can close the browser and turn off the PC.
3. When the version shows **Complete**, open the notebook's page (Your Work → this notebook) → **Output** →
   download `pipeline_a_gemma4-26b_missouri_native_test150.csv`, `pipeline_a_gemma4-26b_hisar_2.5m_rerun.csv`
   and `run_info_repeats.json`.


In [ ]:
# 1. GPU check
!nvidia-smi

In [ ]:
# 2. Install Ollama
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# 3. Start Ollama server
import subprocess, time, requests
subprocess.Popen("ollama serve > /kaggle/working/ollama.log 2>&1", shell=True)
for _ in range(60):
    try:
        requests.get("http://localhost:11434", timeout=2); print("Ollama server is running"); break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("Ollama did not start - see /kaggle/working/ollama.log")
!ollama --version

In [ ]:
# 4. Settings + data
import base64, csv, glob, json, os, re, shutil
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

MODEL = "gemma4:26b"
OLLAMA = "http://localhost:11434"
OUT_DIR = "/kaggle/working"
TMP = "/tmp/crossres"            # degraded collages go here (not into the output)

# ── CHECKPOINT RESTORE: finished rows from an earlier version of this notebook (added as Input) ──
for src in glob.glob("/kaggle/input/**/pipeline_a_gemma4-26b_*.csv", recursive=True):
    dst = os.path.join(OUT_DIR, os.path.basename(src))
    n_src = len(pd.read_csv(src))
    n_dst = len(pd.read_csv(dst)) if os.path.exists(dst) else -1
    if n_src > n_dst:
        shutil.copy(src, dst)
        print(f"Restored checkpoint: {os.path.basename(src)} ({n_src} rows)")

def find_dataset(name):
    hits = [p for p in glob.glob("/kaggle/input/**/collages.csv", recursive=True) if name in p.lower()]
    if not hits:
        raise FileNotFoundError(f"No collages.csv for '{name}' - did you add the gully-collages dataset?")
    folder = os.path.dirname(sorted(hits, key=len)[0])
    df = pd.read_csv(folder + "/collages.csv", dtype={"label": str})
    df["path"] = folder + "/" + df["collage"]
    df = df[df["path"].map(os.path.exists)].reset_index(drop=True)
    print(f"{name}: {len(df)} collages in {folder}")
    return df

missouri = find_dataset("missouri")
hisar = find_dataset("hisar")

# The 150 Missouri test fields: same selection as notebooks 08b and 08c (75 + 75, seed 42)
test150 = (missouri[missouri["label"].isin(["0", "1"])]
           .groupby("label").sample(n=75, random_state=42))
TEST150_IDS = set(test150["tile_id"])
print("test150:", test150["label"].value_counts().to_dict())

In [ ]:
# 5. The question (the 'evidence' prompt, unchanged from notebook 08c) + helpers
PROMPT = ('The image is a collage of {n} overhead images of the SAME agricultural field, taken on different dates. '
          'Each tile has its date or image number in the top-left corner. Ignore any grey "no image" tile or text tile.\n\n'
          'An ephemeral gully is a small erosion channel cut by rainwater runoff, which farmers fill in during tillage. '
          'Many fields have no gully, so only give a high score if you see clear evidence.\n\n'
          'First, list the linear features you can see (if any) and say what each most likely is, '
          'e.g. crop rows, field boundary, road, tractor track, canal, or erosion channel. Then give your score.\n\n'
          'Score from 1 to 5 how likely it is that this field contains an ephemeral gully:\n'
          '1 = almost certainly no, 2 = probably no, 3 = unsure, 4 = probably yes, 5 = almost certainly yes.')
SCHEMA = {"type": "object",
          "properties": {"features": {"type": "string"},
                         "score": {"type": "integer", "enum": [1, 2, 3, 4, 5]}},
          "required": ["features", "score"]}

def pull(model):
    print(f"Downloading {model} ...", flush=True)
    r = requests.post(f"{OLLAMA}/api/pull", json={"model": model, "stream": False}, timeout=3600)
    r.raise_for_status()
    print("  done:", r.json().get("status"))

def ask(model, image_path, n_images):
    with open(image_path, "rb") as f:
        img = base64.b64encode(f.read()).decode()
    body = {"model": model,
            "messages": [{"role": "user", "content": PROMPT.format(n=n_images), "images": [img]}],
            "format": SCHEMA, "stream": False, "think": False,
            "options": {"temperature": 0, "num_ctx": 8192, "seed": 42, "num_predict": 700}}
    r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=900)
    if r.status_code == 400 and "think" in r.text.lower():      # model has no thinking switch
        body.pop("think"); r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=900)
    if r.status_code >= 500:                                     # some models fail with the JSON-format option
        body.pop("format", None); body.pop("think", None)
        body["messages"][0]["content"] += '\n\nAnswer only with JSON: {"features": "...", "score": <1-5>}'
        r = requests.post(f"{OLLAMA}/api/chat", json=body, timeout=900)
    r.raise_for_status()
    raw = r.json()["message"]["content"]
    return parse(raw) + (raw,)

def parse(raw):
    try:
        out = json.loads(raw)
        return int(out["score"]), str(out.get("features", ""))
    except Exception:
        m = re.search(r'"?score"?\s*[:=]\s*"?([1-5])', raw, re.I)
        return (int(m.group(1)) if m else ""), raw

def csv_path(name):
    return f"{OUT_DIR}/pipeline_a_{MODEL.replace(':', '-')}_{name}.csv"

def good_rows(p):
    d = pd.read_csv(p, dtype={"label": str})
    return d[~d["reason"].astype(str).str.startswith("ERROR")]

class TooManyErrors(Exception):
    pass

def run(name, df):
    out_csv = csv_path(name)
    if os.path.exists(out_csv):
        good_rows(out_csv).to_csv(out_csv, index=False)          # drop failed rows so they get retried
    done = set(pd.read_csv(out_csv)["tile_id"]) if os.path.exists(out_csv) else set()
    todo = df[~df["tile_id"].isin(done)]
    new_file = not os.path.exists(out_csv)
    with open(out_csv, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(["tile_id", "label", "score", "reason", "seconds", "raw"])
        errors_in_a_row = 0
        for _, row in tqdm(todo.iterrows(), total=len(todo), desc=name):
            t = time.time()
            try:
                s, reason, raw = ask(MODEL, row["path"], row["n_images"])
                errors_in_a_row = 0
            except Exception as e:
                s, reason, raw = "", f"ERROR: {e}", ""
                errors_in_a_row += 1
                if errors_in_a_row >= 10:
                    raise TooManyErrors(f"10 errors in a row, last: {e}")
            w.writerow([row["tile_id"], row["label"] if pd.notna(row["label"]) else "", s, reason,
                        round(time.time() - t, 1), raw])
            f.flush()
    return pd.read_csv(out_csv, dtype={"label": str})

In [ ]:
# 6. The two repeat runs: original collages, nothing changed
jobs = [("missouri_native_test150", missouri[missouri["tile_id"].isin(TEST150_IDS)].copy()),
        ("hisar_2.5m_rerun", hisar.copy())]
for name, d in jobs:
    print(f"{name}: {len(d)} locations")

In [ ]:
# 7. Run Pipeline A on every condition (the model is downloaded once)
pull(MODEL)

info = {"date": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime()), "model": MODEL,
        "ollama_version": subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip(),
        "gpus": subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                               capture_output=True, text=True).stdout.strip(),
        "options": {"temperature": 0, "num_ctx": 8192, "seed": 42, "num_predict": 700}}
try:
    tags = requests.get(f"{OLLAMA}/api/tags", timeout=30).json().get("models", [])
    info["model_entry"] = [{k: m.get(k) for k in ("name", "digest", "size", "details")} for m in tags
                           if m.get("name", "").startswith(MODEL)]
    show = requests.post(f"{OLLAMA}/api/show", json={"model": MODEL}, timeout=60).json()
    info["details"] = show.get("details")
except Exception as e:
    info["error"] = str(e)
with open(f"{OUT_DIR}/run_info_repeats.json", "w") as f:
    json.dump(info, f, indent=2)
print(json.dumps(info, indent=2))

for name, d in jobs:
    print(f"\n========== {name} ({len(d)} locations) ==========")
    run(name, d)

In [ ]:
# 8. Quick look at the results (the full analysis is done with the labels in the GitHub repository)
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

for name, _ in jobs:
    d = pd.read_csv(csv_path(name), dtype={"label": str})
    d["s"] = pd.to_numeric(d["score"], errors="coerce")
    print(f"\n{name}: {d['s'].notna().sum()} of {len(d)} answered; scores {d['s'].value_counts().sort_index().to_dict()}; "
          f"median {d['seconds'].median():.1f} s per location")
    lab = d[d["label"].isin(["0", "1"]) & d["s"].notna()]
    for part, x in [("all labelled", lab), ("held-out (not test150)", lab[~lab["tile_id"].isin(TEST150_IDS)])]:
        if len(x) and x["label"].nunique() == 2:
            y = x["label"].astype(int)
            print(f"  {part}: AUC {roc_auc_score(y, x['s']):.3f}, balanced accuracy at score >= 3: "
                  f"{balanced_accuracy_score(y, (x['s'] >= 3).astype(int)):.3f}, said gully {(x['s'] >= 3).mean():.0%}")
    if not len(lab):
        print(f"  flagged as gully (score >= 3): {(d['s'] >= 3).sum()} of {d['s'].notna().sum()}")

print("\nFor comparison, the original run (06 Nov): Missouri held-out AUC 0.862, balanced accuracy 0.817; "
      "Hisar flagged 21 of 300.")

## Done — now download

Notebook page → **Output** → download these three files and send them to Claude:

- `pipeline_a_gemma4-26b_missouri_native_test150.csv`
- `pipeline_a_gemma4-26b_hisar_2.5m_rerun.csv`
- `run_info_repeats.json`
